In [1]:
# ============================================================
# CELL 1 — Import Libraries
# ============================================================
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import StandardScaler
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

✅ All libraries imported successfully
pandas version: 3.0.3
numpy version: 2.4.6


In [3]:
# ============================================================
# CELL 2 — Load & Explore the Dataset
# ============================================================
# Load the dataset
df = pd.read_csv('../data/creditcard.csv')

# Basic exploration
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total transactions:     {len(df):,}")
print(f"Total features:         {df.shape[1]}")
print(f"Legitimate (Class 0):   {(df['Class'] == 0).sum():,}")
print(f"Fraudulent (Class 1):   {(df['Class'] == 1).sum():,}")
print(f"Fraud percentage:       {(df['Class'] == 1).mean() * 100:.4f}%")
print()
print("=" * 50)
print("COLUMN NAMES")
print("=" * 50)
print(df.columns.tolist())
print()
print("=" * 50)
print("SAMPLE DATA (first 3 rows)")
print("=" * 50)
print(df.head(3))
print()
print("=" * 50)
print("MISSING VALUES")
print("=" * 50)
print(f"Total missing values: {df.isnull().sum().sum()}")
print("✅ Data loaded and explored successfully")

DATASET OVERVIEW
Total transactions:     284,807
Total features:         31
Legitimate (Class 0):   284,315
Fraudulent (Class 1):   492
Fraud percentage:       0.1727%

COLUMN NAMES
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

SAMPLE DATA (first 3 rows)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.24767

In [4]:
# ============================================================
# CELL 3 — Preprocess Data & Train the Model
# ============================================================

# STEP 1 — Scale Amount and Time columns
# 🧠 WHY: V1-V28 are already scaled by PCA. But Amount and Time
# are raw numbers (€0 to €25,000). ML models perform better when
# all features are on the same scale. StandardScaler transforms
# them to have mean=0 and standard deviation=1.
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])

# STEP 2 — Drop original unscaled columns
df = df.drop(['Amount', 'Time'], axis=1)

# STEP 3 — Separate features (X) and target (y)
# 🧠 WHY: X = what the model LOOKS AT to make a decision
#         y = what the model TRIES TO PREDICT
X = df.drop('Class', axis=1)
y = df['Class']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")

# STEP 4 — Split into training and testing sets
# 🧠 WHY: We train on 80% of data and TEST on the remaining 20%
# that the model has NEVER seen. This tells us how well it works
# on brand new transactions — exactly like real production use.
# stratify=y ensures both sets have the same fraud ratio (0.17%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set size:   {len(X_train):,} transactions")
print(f"Testing set size:    {len(X_test):,} transactions")
print(f"Training fraud cases: {y_train.sum():,}")
print(f"Testing fraud cases:  {y_test.sum():,}")

# STEP 5 — Train the Random Forest Model
# 🧠 WHY these settings:
# n_estimators=100  → 100 decision trees voting together
# class_weight='balanced' → THIS IS THE KEY SETTING that fixes
# our class imbalance problem. It tells the model to pay 
# 578x more attention to fraud cases than legitimate ones
# (because legitimate cases are 578x more common)
# random_state=42 → makes results reproducible every time
# n_jobs=-1 → uses ALL your CPU cores to train faster
print("\n⏳ Training Random Forest model...")
print("This will take 1-3 minutes on your machine...")

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("✅ Model trained successfully!")

Features shape: (284807, 30)
Target shape:   (284807,)
Feature columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_Scaled', 'Time_Scaled']

Training set size:   227,845 transactions
Testing set size:    56,962 transactions
Training fraud cases: 394
Testing fraud cases:  98

⏳ Training Random Forest model...
This will take 1-3 minutes on your machine...
✅ Model trained successfully!


In [5]:
# ============================================================
# CELL 4 — Evaluate Model Performance
# ============================================================

# STEP 1 — Make predictions on the test set
# 🧠 WHY: predict() gives us 0 or 1 (legitimate or fraud)
#         predict_proba() gives us the probability (0.0 to 1.0)
#         e.g. 0.95 means "95% confident this is fraud"
#         We need both for our FastAPI endpoint
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# STEP 2 — Calculate metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MODEL PERFORMANCE REPORT")
print("=" * 50)
print(f"Precision Score:  {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall Score:     {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score:         {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC Score:    {roc_auc:.4f} ({roc_auc*100:.2f}%)")
print()
print("=" * 50)
print("CONFUSION MATRIX")
print("=" * 50)
cm = confusion_matrix(y_test, y_pred)
print(f"True Negatives  (Legit correctly identified):  {cm[0][0]:,}")
print(f"False Positives (Legit wrongly flagged):        {cm[0][1]:,}")
print(f"False Negatives (Fraud missed by model):        {cm[1][0]:,}")
print(f"True Positives  (Fraud correctly caught):       {cm[1][1]:,}")
print()
print("=" * 50)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print()

# STEP 3 — Top 10 most important features
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=" * 50)
print("TOP 10 MOST IMPORTANT FEATURES")
print("=" * 50)
for i, row in feature_importance.head(10).iterrows():
    bar = "█" * int(row['importance'] * 200)
    print(f"{row['feature']:15} {row['importance']:.4f}  {bar}")

print("\n✅ Model evaluation complete!")

MODEL PERFORMANCE REPORT
Precision Score:  0.9186 (91.86%)
Recall Score:     0.8061 (80.61%)
F1 Score:         0.8587 (85.87%)
ROC-AUC Score:    0.9518 (95.18%)

CONFUSION MATRIX
True Negatives  (Legit correctly identified):  56,857
False Positives (Legit wrongly flagged):        7
False Negatives (Fraud missed by model):        19
True Positives  (Fraud correctly caught):       79

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.92      0.81      0.86        98

    accuracy                           1.00     56962
   macro avg       0.96      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962


TOP 10 MOST IMPORTANT FEATURES
V14             0.2170  ███████████████████████████████████████████
V10             0.1118  ██████████████████████
V4              0.1103  ██████████████████████
V17             0.0861  █████████████████
V12             0.08

In [6]:
# ============================================================
# CELL 5 — Save the Trained Model & Scaler
# ============================================================

# 🧠 WHY are we saving the model?
# Training takes 1-3 minutes every time. In production, we NEVER
# retrain the model on every API request. Instead we:
# 1. Train ONCE and save to a file (this cell)
# 2. Load the file instantly when FastAPI starts (next phase)
# This is called "model serialization" — a key MLOps concept

# 🧠 WHY save the scaler too?
# When a new transaction comes in through our API, we must scale
# its Amount and Time values using the EXACT SAME scaler we used
# during training. If we create a new scaler, it will have
# different parameters and our predictions will be wrong.
# This is called "training-serving skew" — another key concept.

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the trained model
model_path = '../models/fraud_model.pkl'
joblib.dump(model, model_path)
print(f"✅ Model saved to: {model_path}")

# Save the scaler
scaler_path = '../models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler saved to: {scaler_path}")

# Save feature names so our API knows the exact column order
feature_names = X.columns.tolist()
feature_path = '../models/feature_names.pkl'
joblib.dump(feature_names, feature_path)
print(f"✅ Feature names saved to: {feature_path}")

# Verify all files were saved correctly
print()
print("=" * 50)
print("SAVED MODEL FILES")
print("=" * 50)
for fname in ['fraud_model.pkl', 'scaler.pkl', 'feature_names.pkl']:
    fpath = f'../models/{fname}'
    size = os.path.getsize(fpath) / (1024 * 1024)
    print(f"{fname:25} {size:.2f} MB")

# Quick sanity check — reload and predict one transaction
print()
print("=" * 50)
print("SANITY CHECK — RELOAD & PREDICT")
print("=" * 50)
loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)
sample = X_test.iloc[0:1]
sample_pred = loaded_model.predict(sample)[0]
sample_proba = loaded_model.predict_proba(sample)[0][1]
actual = y_test.iloc[0]
print(f"Sample prediction:  {'FRAUD' if sample_pred == 1 else 'LEGITIMATE'}")
print(f"Fraud probability:  {sample_proba:.4f} ({sample_proba*100:.2f}%)")
print(f"Actual label:       {'FRAUD' if actual == 1 else 'LEGITIMATE'}")
print(f"Prediction correct: {'✅ YES' if sample_pred == actual else '❌ NO'}")
print()
print("🎉 Phase 3 Complete! Model is trained, evaluated and saved!")
print("Next: Build the FastAPI backend to serve this model")

✅ Model saved to: ../models/fraud_model.pkl
✅ Scaler saved to: ../models/scaler.pkl
✅ Feature names saved to: ../models/feature_names.pkl

SAVED MODEL FILES
fraud_model.pkl           5.07 MB
scaler.pkl                0.00 MB
feature_names.pkl         0.00 MB

SANITY CHECK — RELOAD & PREDICT
Sample prediction:  LEGITIMATE
Fraud probability:  0.0000 (0.00%)
Actual label:       LEGITIMATE
Prediction correct: ✅ YES

🎉 Phase 3 Complete! Model is trained, evaluated and saved!
Next: Build the FastAPI backend to serve this model
